# Day 21: Autoscaling — Concurrency, Batching & Cold Starts
> *Inference Engineering* — Chapter 7.2 | Philip Kiely, Baseten Books 2026

**Layer:** Infrastructure | **Prerequisite:** Day 20 (Containers)


## Concept Overview

Inference autoscaling adds or removes GPU replicas based on traffic signals. The key metrics are: concurrency (simultaneous in-flight requests), queue depth, and GPU utilization. Scaling on concurrency is better than CPU-style scaling on utilization — a GPU at 30% utilization may be serving 10 concurrent requests perfectly well. Cold start latency (time to load a model from disk to GPU memory) is the primary challenge: a 70B model may take 60-90 seconds to load, during which incoming traffic queues up. Strategies to mitigate cold starts: model warm pools, pre-loading, and predictive scaling based on time-of-day patterns.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import time, random

print('Autoscaling simulation environment loaded')


## 1. Scaling Triggers: Concurrency vs Utilization

GPU utilization is a poor scaling signal for LLM inference — it's consistently high during decode regardless of load level.


In [ ]:
# Simulate why GPU utilization is a bad scaling metric
np.random.seed(42)
T = 100  # time steps

# Scenario: requests arrive at variable rate
arrival_rate = np.concatenate([
    np.ones(25) * 2,   # low load
    np.ones(25) * 8,   # high load
    np.ones(25) * 20,  # overload
    np.ones(25) * 4,   # medium
])

MAX_CONCURRENCY = 10
queue = deque()
active = []
times = range(T)
concurrency_series = []
utilization_series = []
queue_depth_series = []

for t in times:
    # New arrivals
    n_new = np.random.poisson(arrival_rate[t])
    for _ in range(n_new):
        queue.append({'arrival': t, 'duration': np.random.randint(3, 12)})

    # Fill active slots
    while queue and len(active) < MAX_CONCURRENCY:
        active.append(queue.popleft())

    # GPU utilization: high whenever any request is active
    util = 1.0 if active else 0.0  # GPU is "busy" whenever anything is running

    # Tick active requests
    active = [r for r in active if r['duration'] > 0]
    for r in active:
        r['duration'] -= 1

    concurrency_series.append(len(active))
    utilization_series.append(util * 100)
    queue_depth_series.append(len(queue))

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(times, concurrency_series, label='Concurrency')
axes[0].axhline(MAX_CONCURRENCY * 0.8, color='red', linestyle='--', label='Scale threshold (80%)')
axes[0].set_ylabel('Active Requests'); axes[0].legend()
axes[1].plot(times, utilization_series, label='GPU Utilization %', color='orange')
axes[1].set_ylabel('GPU Util %'); axes[1].legend()
axes[1].set_ylim(0, 110)
axes[2].fill_between(times, queue_depth_series, label='Queue Depth', alpha=0.5, color='red')
axes[2].set_ylabel('Queue Depth'); axes[2].set_xlabel('Time'); axes[2].legend()
plt.suptitle('Autoscaling Metrics: Concurrency vs GPU Utilization')
plt.tight_layout()
plt.savefig('autoscaling_metrics.png', dpi=100, bbox_inches='tight')
plt.show()
print('Note: GPU utilization stays high (100%) even during overload — concurrency and queue depth are better signals')


## 2. Cold Start Latency Model

Model load time is the dominant cold start cost. Understanding the components helps design mitigation strategies.


In [ ]:
def model_load_time_estimate(model_params_b, disk_bw_gb_s=3.0, gpu_bw_gb_s=10.0,
                             tokenizer_s=1.0, kv_alloc_s=2.0, dtype_bytes=2):
    """Estimate model cold start time."""
    model_gb = model_params_b * dtype_bytes
    disk_load_s = model_gb / disk_bw_gb_s
    gpu_transfer_s = model_gb / gpu_bw_gb_s
    compile_s = model_params_b * 0.5  # rough: 0.5s per billion params for TRT compile
    total_s = disk_load_s + gpu_transfer_s + tokenizer_s + kv_alloc_s
    return {
        'disk_load': disk_load_s,
        'gpu_transfer': gpu_transfer_s,
        'tokenizer': tokenizer_s,
        'kv_alloc': kv_alloc_s,
        'total': total_s,
    }

models = [(1, 'Llama-3.2-1B'), (8, 'Llama-3-8B'), (70, 'Llama-3-70B'), (405, 'Llama-3-405B')]

print('Model cold start time estimates:')
print(f'{"Model":<20} {"Disk Load":>12} {"GPU Xfer":>10} {"Total":>8}')
for params, name in models:
    t = model_load_time_estimate(params)
    print(f'{name:<20} {t["disk_load"]:>11.1f}s {t["gpu_transfer"]:>9.1f}s {t["total"]:>7.1f}s')

print('\nCold start mitigation strategies:')
strategies = [
    ('Model warm pool',       'Keep N idle replicas pre-loaded; route to them immediately'),
    ('Predictive scaling',    'Scale up before traffic spikes (time-of-day, cron)'),
    ('Faster storage',        'NVMe SSD or RAM disk for model weights'),
    ('Lazy loading',          'Load only the requested layers (attention first, FFN second)'),
    ('Shared model volumes',  'Multiple containers sharing same model files via NFS/S3FS'),
]
for name, desc in strategies:
    print(f'  {name:<25} {desc}')


## 3. Autoscaling Policy

A simple reactive autoscaling policy based on concurrency queue depth.


In [ ]:
class InferenceAutoscaler:
    def __init__(self, min_replicas=1, max_replicas=8,
                 target_concurrency=8, cold_start_s=30):
        self.replicas = min_replicas
        self.min_replicas = min_replicas
        self.max_replicas = max_replicas
        self.target_concurrency = target_concurrency
        self.cold_start_s = cold_start_s
        self.pending_replicas = 0  # starting up
        self.scale_events = []

    def tick(self, t, current_concurrency, queue_depth):
        # Desired replicas based on concurrency
        desired = max(self.min_replicas,
                     int(np.ceil(current_concurrency / self.target_concurrency)))
        # Emergency scale if queue is building
        if queue_depth > self.target_concurrency * 2:
            desired = min(self.max_replicas, desired + 1)

        desired = min(desired, self.max_replicas)
        effective = self.replicas  # cold-start replicas not yet effective

        if desired > effective + self.pending_replicas:
            added = desired - effective - self.pending_replicas
            self.pending_replicas += added
            self.scale_events.append(('scale_up', t, added))
        elif desired < effective and self.pending_replicas == 0:
            removed = effective - desired
            self.replicas -= removed
            self.scale_events.append(('scale_down', t, removed))

        # Simulate replicas coming online after cold start
        if self.pending_replicas > 0 and t % int(self.cold_start_s) == 0:
            came_online = self.pending_replicas
            self.replicas += came_online
            self.pending_replicas = 0

        return self.replicas

scaler = InferenceAutoscaler(min_replicas=1, max_replicas=8, target_concurrency=8)

replica_history = []
for t, (conc, qdepth) in enumerate(zip(concurrency_series, queue_depth_series)):
    r = scaler.tick(t, conc, qdepth)
    replica_history.append(r)

plt.figure(figsize=(12, 4))
plt.subplot(2, 1, 1)
plt.plot(concurrency_series, label='Concurrency')
plt.plot(queue_depth_series, label='Queue Depth', alpha=0.7)
plt.legend(); plt.ylabel('Count')
plt.subplot(2, 1, 2)
plt.step(range(T), replica_history, where='post', label='Replicas', color='green')
plt.ylabel('Replica Count'); plt.xlabel('Time')
plt.legend()
plt.tight_layout()
plt.savefig('autoscaler_response.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Scale events: {scaler.scale_events[:5]}')


## Experiments: Try These

1. **Measure actual cold start**: SSH to spark-01 and time `vllm serve meta-llama/Llama-3.2-1B-Instruct` from cold. How long until the first request is served?
2. **Hysteresis**: Add a cooldown period to the autoscaler (don't scale down for N seconds after a scale-up). How does this affect steady-state replica count under bursty traffic?
3. **Predictive scaling**: Implement a scaler that reads historical traffic patterns and pre-scales 5 minutes before expected peaks. Test on a synthetic daily pattern.


## Key Takeaways

- Scale on concurrency (in-flight requests) or queue depth, not GPU utilization — GPU is saturated at any non-zero load level.
- Cold start latency scales with model size and disk bandwidth — a 70B model takes 60-90 seconds to load, creating a scaling lag.
- Model warm pools (idle pre-loaded replicas) are the most reliable cold start mitigation — at the cost of idle GPU spend.
- Autoscaling hysteresis prevents thrashing: add a cooldown period and require sustained load before scaling decisions.

## References
- *Inference Engineering* Ch 7.2 — Philip Kiely, Baseten Books 2026
- Kubernetes HPA documentation
- KEDA (Kubernetes Event-Driven Autoscaling) documentation
